In [1]:
import pandas as pd

books = pd.read_csv("../data/books_cleaned.csv")

In [2]:
books["categories"].value_counts().reset_index()

,categories,count
0,Exhibitions,232
1,American literature,221
2,Children's fiction,150
3,English literature,139
4,"Fiction, romance, general",133
...,...,...
91624,Australian Creative nonfiction,1
91625,Budgerigar;History,1
91626,Families;Fiction;Orphans;Loneliness,1
91627,History;Biographies;Memoirs,1


In [3]:
# get rid of categories with less than 50 books
books["categories"].value_counts().reset_index().query("count > 50")

,categories,count
0,Exhibitions,232
1,American literature,221
2,Children's fiction,150
3,English literature,139
4,"Fiction, romance, general",133
5,Literature,110
6,Artistic Photography,76
7,"Fiction, general",66
8,Poetry (poetic works by one author),58
9,Canadian poetry,57


In [4]:
import re

# OL puts many subjects into one ';'-joined string so scan for keywords. 
# "nonfiction" contains "fiction", so nonfiction tested before fiction.

# genre hints consulted when no explicit fiction/nonfiction label is present.
FICTION_HINTS = (
    "fantasy", "romance", "thriller", "mystery", "horror", "science fiction",
    "short stories", "fairy tale", "graphic novel", "comic", "detective",
    "adventure", "poetry", "drama", "novel",
)
NONFICTION_HINTS = (
    "biography", "autobiography", "history", "philosophy", "religion", "self-help",
    "cooking", "travel", "business", "psychology", "memoir", "true crime", "essays",
    "science", "reference", "health",
)

def simplify_categories(cats):
    if not isinstance(cats, str):
        return None
    c = cats.lower()

    # children's books collapse to one bucket: the fiction/nonfiction sub-split was sparse
    # (Children's Nonfiction ~1%) and noisy, and the juvenile signal is far more reliable
    # than that sub-decision. Checked first, so we don't even need to resolve fiction kind.
    if ("juvenile" in c) or ("children" in c):
        return "Children's"

    if re.search(r"non-?fiction", c):              # 1: explicit labels
        return "Nonfiction"
    if re.search(r"\bfiction\b", c):
        return "Fiction"
    if any(h in c for h in FICTION_HINTS):         # 2: genre hints
        return "Fiction"
    if any(h in c for h in NONFICTION_HINTS):
        return "Nonfiction"
    return None                                    # leave for the BART backfill below

books["simple_categories"] = books["categories"].apply(simplify_categories)
books["simple_categories"].value_counts(dropna=False)

simple_categories
Nonfiction    36331
NaN           30196
Fiction       22715
Children's    11387
Name: count, dtype: int64

In [5]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9781411668980,"The Galaxii Series: Book 1 ""Blachart""",Christina Engela,"Fiction, romance, general",Action! Adventure! Space Opera! Life hardly ev...,https://covers.openlibrary.org/b/id/14313750-L...,2018,NaN,280.0,"The Galaxii Series: Book 1 ""Blachart""",9781411668980 Action! Adventure! Space Opera! ...,Fiction
1,9780226575087,Lost Mars,Michael Ashley,"Fiction;Science Fiction;Fiction, science ficti...",Ten short stories from the golden age of scien...,https://covers.openlibrary.org/b/id/13133252-L...,2018,NaN,302.0,Lost Mars: stories from the golden age of the ...,9780226575087 Ten short stories from the golde...,Fiction
2,9780062467874,Villain,Michael Grant,Juvenile fiction;Fiction;Supernatural;Horror s...,MONSTER. VILLAIN. HERO. WHICH SUPERCREATURE WI...,https://covers.openlibrary.org/b/id/8814378-L.jpg,2018,NaN,324.0,Villain,9780062467874 MONSTER. VILLAIN. HERO. WHICH SU...,Children's
3,9780062930484,The ABC Murders,Agatha Christie,Fiction;Mystery;Agatha Christie;Hercule Poirot...,"There's a serial killer on the loose, bent on ...",https://covers.openlibrary.org/b/id/-1-L.jpg,2019,NaN,272.0,The ABC Murders: A Hercule Poirot Mystery,9780062930484 There's a serial killer on the l...,Fiction
4,9781632365804,Welcome to the ballroom,Tomo Takeuchi,Competitions;Ballroom dancing;Dance;Ballroom d...,"""Through sheer force of will, Tatara and China...",NaN,2018,NaN,NaN,Welcome to the ballroom,"9781632365804 ""Through sheer force of will, Ta...",Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...
100624,9781925704259,Dizzy limits,Noëlle Janaczewska,Australian Creative nonfiction,When conventional approaches to writing about ...,NaN,2020,NaN,440.0,Dizzy limits: recent experiments in Australian...,9781925704259 When conventional approaches to ...,Nonfiction
100625,9780642279606,Flight of the Budgerigar,Penny Olsen,Budgerigar;History,Taking the reader from the Dreaming to the col...,NaN,2021,NaN,251.0,Flight of the Budgerigar: an illustrated history,9780642279606 Taking the reader from the Dream...,Nonfiction
100626,9781988254685,Walls of the cave,Syr Ruus,Families;Fiction;Orphans;Loneliness,"""A writer of unknown gender, an orphan, brough...",NaN,2019,NaN,103.0,Walls of the cave,"9781988254685 ""A writer of unknown gender, an ...",Fiction
100627,9788198859686,The Struggle for Europe,NaN,History;Biographies;Memoirs,"First published in 1952, ‘The Struggle for Eur...",https://covers.openlibrary.org/b/id/15219838-L...,2025,NaN,NaN,The Struggle for Europe,"9788198859686 First published in 1952, ‘The St...",Nonfiction


In [6]:
books[~(books["simple_categories"].isna())]

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9781411668980,"The Galaxii Series: Book 1 ""Blachart""",Christina Engela,"Fiction, romance, general",Action! Adventure! Space Opera! Life hardly ev...,https://covers.openlibrary.org/b/id/14313750-L...,2018,NaN,280.0,"The Galaxii Series: Book 1 ""Blachart""",9781411668980 Action! Adventure! Space Opera! ...,Fiction
1,9780226575087,Lost Mars,Michael Ashley,"Fiction;Science Fiction;Fiction, science ficti...",Ten short stories from the golden age of scien...,https://covers.openlibrary.org/b/id/13133252-L...,2018,NaN,302.0,Lost Mars: stories from the golden age of the ...,9780226575087 Ten short stories from the golde...,Fiction
2,9780062467874,Villain,Michael Grant,Juvenile fiction;Fiction;Supernatural;Horror s...,MONSTER. VILLAIN. HERO. WHICH SUPERCREATURE WI...,https://covers.openlibrary.org/b/id/8814378-L.jpg,2018,NaN,324.0,Villain,9780062467874 MONSTER. VILLAIN. HERO. WHICH SU...,Children's
3,9780062930484,The ABC Murders,Agatha Christie,Fiction;Mystery;Agatha Christie;Hercule Poirot...,"There's a serial killer on the loose, bent on ...",https://covers.openlibrary.org/b/id/-1-L.jpg,2019,NaN,272.0,The ABC Murders: A Hercule Poirot Mystery,9780062930484 There's a serial killer on the l...,Fiction
4,9781632365804,Welcome to the ballroom,Tomo Takeuchi,Competitions;Ballroom dancing;Dance;Ballroom d...,"""Through sheer force of will, Tatara and China...",NaN,2018,NaN,NaN,Welcome to the ballroom,"9781632365804 ""Through sheer force of will, Ta...",Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...
100624,9781925704259,Dizzy limits,Noëlle Janaczewska,Australian Creative nonfiction,When conventional approaches to writing about ...,NaN,2020,NaN,440.0,Dizzy limits: recent experiments in Australian...,9781925704259 When conventional approaches to ...,Nonfiction
100625,9780642279606,Flight of the Budgerigar,Penny Olsen,Budgerigar;History,Taking the reader from the Dreaming to the col...,NaN,2021,NaN,251.0,Flight of the Budgerigar: an illustrated history,9780642279606 Taking the reader from the Dream...,Nonfiction
100626,9781988254685,Walls of the cave,Syr Ruus,Families;Fiction;Orphans;Loneliness,"""A writer of unknown gender, an orphan, brough...",NaN,2019,NaN,103.0,Walls of the cave,"9781988254685 ""A writer of unknown gender, an ...",Fiction
100627,9788198859686,The Struggle for Europe,NaN,History;Biographies;Memoirs,"First published in 1952, ‘The Struggle for Eur...",https://covers.openlibrary.org/b/id/15219838-L...,2025,NaN,NaN,The Struggle for Europe,"9788198859686 First published in 1952, ‘The St...",Nonfiction


In [7]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="mps")
# mps is the Apple Mac specific GPU

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [8]:
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[0]
# resetting the index ensure indexes correlate to the classified books not the original database

In [9]:
pipe(sequence, fiction_categories)
# returns probability that the book is within each category

{'sequence': 'Action! Adventure! Space Opera! Life hardly ever turns out the way we expect it to, and for Mykl d’Angelo, skipper and owner of the loderunner Pegasus, it had just taken a bad turn for terrible. Due a minor misunderstanding, his crew had stolen the ship’s only shuttle, leaving him and two others behind to crew the ailing ship on their own. As if that weren’t bad enough, just a few hours later the rickety old ship’s stardrive exploded in the middle of the middle of nowhere, killing Mykl’s two remaining crewmen. Marooned alone in deep space, Mykl d’Angelo counted his blessings, offered prayers to any gods who specialized in miracles, and prepared to await (A) seemingly unlikely rescue, or (B) a lingering death… Rescue takes place, but at a price – as the Antares, a cruiser dispatched to investigate the mysterious silence of a remote starbase crosses paths with a legendary and fearsome Corsair – a man whose name sent shivers down the spines of lesser mortals. Blachart… Blach

In [10]:
import numpy as np

max_index = np.argmax(pipe(sequence, fiction_categories)["scores"])
max_label = pipe(sequence, fiction_categories)["labels"][max_index]
max_label

'Fiction'

In [11]:
def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"]) # yields the index of the highest probability
    max_label = predictions["labels"][max_index]
    return max_label

In [12]:
# Testing how good the model is using a sizable sample
from tqdm import tqdm
# tqdm is a library used to add progress bars to loops

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 100)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Fiction"]


  0%|          | 0/100 [00:00<?, ?it/s]


  1%|          | 1/100 [00:00<00:14,  6.74it/s]


  2%|▏         | 2/100 [00:00<00:41,  2.34it/s]


  3%|▎         | 3/100 [00:00<00:31,  3.12it/s]


  4%|▍         | 4/100 [00:01<00:44,  2.14it/s]

...


 94%|█████████▍| 94/100 [00:15<00:00,  8.12it/s]


 95%|█████████▌| 95/100 [00:15<00:00,  8.28it/s]


 97%|█████████▋| 97/100 [00:15<00:00,  9.65it/s]


 99%|█████████▉| 99/100 [00:15<00:00,  9.16it/s]


100%|██████████| 100/100 [00:16<00:00,  6.23it/s]

In [13]:
for i in tqdm(range(0, 100)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Nonfiction"]


  0%|          | 0/100 [00:00<?, ?it/s]


  1%|          | 1/100 [00:00<00:11,  8.94it/s]


  2%|▏         | 2/100 [00:00<00:24,  3.97it/s]


  3%|▎         | 3/100 [00:00<00:24,  3.96it/s]


  5%|▌         | 5/100 [00:01<00:20,  4.56it/s]

...


 95%|█████████▌| 95/100 [00:15<00:00,  6.05it/s]


 96%|█████████▌| 96/100 [00:15<00:00,  6.09it/s]


 98%|█████████▊| 98/100 [00:16<00:00,  6.77it/s]


 99%|█████████▉| 99/100 [00:16<00:00,  7.23it/s]


100%|██████████| 100/100 [00:16<00:00,  6.14it/s]

In [14]:
predictions_df = pd.DataFrame({"actual_categories": actual_cats, "predicted_categories": predicted_cats})
predictions_df

,actual_categories,predicted_categories
0,Fiction,Fiction
1,Fiction,Fiction
2,Fiction,Fiction
3,Fiction,Fiction
4,Fiction,Fiction
...,...,...
195,Nonfiction,Nonfiction
196,Nonfiction,Nonfiction
197,Nonfiction,Nonfiction
198,Nonfiction,Nonfiction


In [15]:
predictions_df["correct_prediction"] = (
    np.where(predictions_df["actual_categories"] == predictions_df["predicted_categories"], 1, 0)
)

In [ ]:
# Percentage accuracy of the BART zero-shot Fiction/Nonfiction backfill on the 200-book labelled sample.
predictions_df["correct_prediction"].sum() / len(predictions_df)

np.float64(0.8)

In [17]:
# make a subset of the database with the simple category missing
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [18]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    isbns += [missing_cats["isbn13"][i]]


  0%|          | 0/30196 [00:00<?, ?it/s]


  0%|          | 1/30196 [00:00<1:25:48,  5.86it/s]


  0%|          | 2/30196 [00:00<1:19:01,  6.37it/s]


  0%|          | 3/30196 [00:00<1:08:04,  7.39it/s]


  0%|          | 4/30196 [00:00<1:14:03,  6.79it/s]

...


100%|█████████▉| 30191/30196 [1:02:29<00:00,  7.69it/s]


100%|█████████▉| 30193/30196 [1:02:29<00:00,  9.50it/s]


100%|█████████▉| 30195/30196 [1:02:29<00:00,  7.21it/s]


100%|██████████| 30196/30196 [1:02:29<00:00,  7.57it/s]


100%|██████████| 30196/30196 [1:02:29<00:00,  8.05it/s]

In [19]:
missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_categories": predicted_cats})

In [20]:
missing_predicted_df

,isbn13,predicted_categories
0,9780399579417,Nonfiction
1,9781782495499,Nonfiction
2,9781611720433,Nonfiction
3,9780520290686,Nonfiction
4,9781633693104,Nonfiction
...,...,...
30191,9781838955267,Nonfiction
30192,9780192882233,Nonfiction
30193,9782370742636,Nonfiction
30194,9789462905740,Nonfiction


In [21]:
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left") 
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_categories"], books["simple_categories"])
# merge into books - when category is missing use predicted, if simple category there then use that
books = books.drop(columns = ["predicted_categories"])

In [22]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9781411668980,"The Galaxii Series: Book 1 ""Blachart""",Christina Engela,"Fiction, romance, general",Action! Adventure! Space Opera! Life hardly ev...,https://covers.openlibrary.org/b/id/14313750-L...,2018,NaN,280.0,"The Galaxii Series: Book 1 ""Blachart""",9781411668980 Action! Adventure! Space Opera! ...,Fiction
1,9780226575087,Lost Mars,Michael Ashley,"Fiction;Science Fiction;Fiction, science ficti...",Ten short stories from the golden age of scien...,https://covers.openlibrary.org/b/id/13133252-L...,2018,NaN,302.0,Lost Mars: stories from the golden age of the ...,9780226575087 Ten short stories from the golde...,Fiction
2,9780062467874,Villain,Michael Grant,Juvenile fiction;Fiction;Supernatural;Horror s...,MONSTER. VILLAIN. HERO. WHICH SUPERCREATURE WI...,https://covers.openlibrary.org/b/id/8814378-L.jpg,2018,NaN,324.0,Villain,9780062467874 MONSTER. VILLAIN. HERO. WHICH SU...,Children's
3,9780062930484,The ABC Murders,Agatha Christie,Fiction;Mystery;Agatha Christie;Hercule Poirot...,"There's a serial killer on the loose, bent on ...",https://covers.openlibrary.org/b/id/-1-L.jpg,2019,NaN,272.0,The ABC Murders: A Hercule Poirot Mystery,9780062930484 There's a serial killer on the l...,Fiction
4,9781632365804,Welcome to the ballroom,Tomo Takeuchi,Competitions;Ballroom dancing;Dance;Ballroom d...,"""Through sheer force of will, Tatara and China...",NaN,2018,NaN,NaN,Welcome to the ballroom,"9781632365804 ""Through sheer force of will, Ta...",Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...
100624,9781925704259,Dizzy limits,Noëlle Janaczewska,Australian Creative nonfiction,When conventional approaches to writing about ...,NaN,2020,NaN,440.0,Dizzy limits: recent experiments in Australian...,9781925704259 When conventional approaches to ...,Nonfiction
100625,9780642279606,Flight of the Budgerigar,Penny Olsen,Budgerigar;History,Taking the reader from the Dreaming to the col...,NaN,2021,NaN,251.0,Flight of the Budgerigar: an illustrated history,9780642279606 Taking the reader from the Dream...,Nonfiction
100626,9781988254685,Walls of the cave,Syr Ruus,Families;Fiction;Orphans;Loneliness,"""A writer of unknown gender, an orphan, brough...",NaN,2019,NaN,103.0,Walls of the cave,"9781988254685 ""A writer of unknown gender, an ...",Fiction
100627,9788198859686,The Struggle for Europe,NaN,History;Biographies;Memoirs,"First published in 1952, ‘The Struggle for Eur...",https://covers.openlibrary.org/b/id/15219838-L...,2025,NaN,NaN,The Struggle for Europe,"9788198859686 First published in 1952, ‘The St...",Nonfiction


In [23]:
books.to_csv("../data/books_with_categories.csv", index=False)